# Variable Transforms

Four wrangling operations in one notebook:

1. **Binary columns to `#multi`** — collapse several 0/1 flag columns into a single pipe-separated multi-value column
2. **`#multi` to binary dummies** — expand a multi-value column into one 0/1 column per unique value
3. **Recode ordinal/categorical** — remap values using a user-defined mapping
4. **Normalize numeric** — min-max or z-score scaling, output as `ColName_norm#number`

Run only the sections you need.

In [ ]:
# ── Colab bootstrap (no-op on Binder / JupyterHub) ────────────────────────
import os, sys
if "google.colab" in sys.modules:
    if not os.path.isfile("/tmp/suave-nb/helpers/suave_utils.py"):
        import subprocess
        _r = subprocess.run(
            ["git", "clone", "--depth=1", "https://github.com/izaslavsky/suave-notebooks.git", "/tmp/suave-nb"],
            capture_output=True, text=True
        )
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        raise RuntimeError('No SuAVE params. Open via SuAVEDispatch.ipynb first.')
else:
    # Colab: try Drive/cache silently; credentials cell below handles fallback
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: enter credentials if Drive had no session ----------------------
if su.ENV.colab:
    _ref = [_p]
    su.colab_credentials(_ref)
    _p = _ref[0]
else:
    su._skipped('Colab only')


In [ ]:
# ── Variables used throughout this notebook ────────────────────────────────
if _p is None:
    raise RuntimeError('Parameters not loaded. Run the cell above to enter credentials.')
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np

df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

---
## Operation 1: Binary columns → `#multi`

Select several 0/1 columns. For each row, the output column will contain the pipe-separated names of columns that equal `1`.

In [ ]:
bin_cols = [c for c in df.columns if '#number' not in c and '#img' not in c
            and '#date' not in c]

op1_cols = widgets.SelectMultiple(
    options=df.columns.tolist(), description='Binary columns:',
    rows=8, layout=widgets.Layout(width='70%', height='180px')
)
op1_name = widgets.Text(
    value='Flags#multi', description='New column name:',
    layout=widgets.Layout(width='60%')
)
printmd("**Select the 0/1 indicator columns to collapse:**")
display(op1_cols, op1_name)

In [ ]:
selected_bin = list(op1_cols.value)
new_col_name = op1_name.value.strip() or 'Flags#multi'
if not new_col_name.endswith('#multi'):
    new_col_name += '#multi'

def _to_multi(row):
    parts = []
    for c in selected_bin:
        try:
            if float(row[c]) == 1:
                parts.append(c.split('#')[0])
        except (ValueError, TypeError):
            pass
    return '|'.join(parts) if parts else None

df[new_col_name] = df.apply(_to_multi, axis=1)
printmd(f"**Column `{new_col_name}` created. Sample:**")
display(df[selected_bin + [new_col_name]].head(5))

---
## Operation 2: `#multi` → binary dummies

In [ ]:
multi_cols = [c for c in df.columns if '#multi' in c]

if not multi_cols:
    printmd('*No `#multi` columns found in this survey.*')
else:
    op2_col = widgets.Dropdown(
        options=multi_cols, description='#multi column:',
        layout=widgets.Layout(width='60%')
    )
    display(op2_col)

In [ ]:
if multi_cols:
    src = op2_col.value
    all_vals = set()
    for cell in df[src].dropna():
        all_vals.update(v.strip() for v in str(cell).split('|') if v.strip())

    new_cols = []
    for val in sorted(all_vals):
        col_name = f"{val}#number"
        df[col_name] = df[src].apply(
            lambda cell: 1 if pd.notna(cell) and val in [v.strip() for v in str(cell).split('|')] else 0
        )
        new_cols.append(col_name)

    printmd(f"**Added {len(new_cols)} dummy columns from `{src}`:**")
    display(df[new_cols].head(5))

---
## Operation 3: Recode values

Enter a mapping as `old_value: new_value` pairs, one per line.

In [ ]:
all_cols = df.columns.tolist()
op3_col = widgets.Dropdown(
    options=all_cols, description='Column:',
    layout=widgets.Layout(width='60%')
)
op3_mapping = widgets.Textarea(
    value='1: Low\n2: Medium\n3: High',
    description='Mapping:',
    layout=widgets.Layout(width='60%', height='100px')
)
op3_new_name = widgets.Text(
    value='', placeholder='Leave blank to overwrite source column',
    description='New column:', layout=widgets.Layout(width='60%')
)
display(op3_col, op3_mapping, op3_new_name)

In [ ]:
src_col  = op3_col.value
out_name = op3_new_name.value.strip() or src_col

mapping = {}
for line in op3_mapping.value.splitlines():
    if ':' in line:
        k, v = line.split(':', 1)
        mapping[k.strip()] = v.strip()

df[out_name] = df[src_col].astype(str).map(mapping).fillna(df[src_col])
printmd(f"**Recoded `{src_col}` → `{out_name}` using {len(mapping)} mappings. Sample:**")
display(df[[src_col, out_name]].drop_duplicates().head(10))

---
## Operation 4: Normalize numeric column

In [ ]:
num_cols = [c for c in df.columns if '#number' in c]
if not num_cols:
    printmd('*No #number columns available.*')
else:
    op4_col = widgets.Dropdown(
        options=num_cols, description='Column:',
        layout=widgets.Layout(width='60%')
    )
    op4_method = widgets.RadioButtons(
        options=['min-max (0–1)', 'z-score'],
        description='Method:'
    )
    display(op4_col, op4_method)

In [ ]:
if num_cols:
    src_col = op4_col.value
    base    = src_col.split('#')[0]
    out_col = f"{base}_norm#number"

    vals = pd.to_numeric(df[src_col], errors='coerce')

    if 'min-max' in op4_method.value:
        mn, mx = vals.min(), vals.max()
        df[out_col] = ((vals - mn) / (mx - mn)).round(6)
        printmd(f"**Min-max normalized `{src_col}` → `{out_col}` (range 0–1).**")
    else:
        mu, sigma = vals.mean(), vals.std()
        df[out_col] = ((vals - mu) / sigma).round(6)
        printmd(f"**Z-score normalized `{src_col}` → `{out_col}` (mean 0, std 1).**")

    display(df[[src_col, out_col]].describe().round(4))

---
## Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
#Input survey name
survey_name = input('Enter survey name: ')
printmd('**Survey Name:** ' + survey_name)
